In [1]:
# The following code will only execute
# successfully when compression is complete

import kagglehub

path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

100%|██████████| 6.72G/6.72G [05:10<00:00, 23.2MB/s]  

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/1


In [2]:
!pip install torch==2.5.0 torchvision==0.20.0 torchaudio==2.5.0 --index-url https://download.pytorch.org/whl/cu124

Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 908.2/908.2 MB 524.1 kB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 116.7 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.0 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 84.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 104.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9

In [3]:
!git clone https://github.com/Omid-Nejati/MedViTV2.git

Cloning into 'MedViTV2'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (77/77), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 189 (delta 54), reused 28 (delta 26), pack-reused 112 (from 1)
Receiving objects: 100% (189/189), 3.87 MiB | 9.14 MiB/s, done.
Resolving deltas: 100% (66/66), done.


In [4]:
!pip install natten==0.17.3+torch250cu124 -f https://shi-labs.com/natten/wheels/ --trusted-host shi-labs.com

Looking in links: https://shi-labs.com/natten/wheels/
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.5/474.5 MB 3.1 MB/s eta 0:00:0000:0100:01


In [5]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")

PyTorch version: 2.5.0+cu124
CUDA version: 12.4


In [6]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        return image, localiser, label


train_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((336, 336)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), transform=train_transform)
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), transform=test_transform)
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), transform=test_transform)

labels = df_train['label'].values.astype(int)

class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

n_labels = df_train['label'].nunique()





In [ ]:
%cd /root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/MedViTV2-main
!pip install -r requirements.txt
from MedViT import MedViT

[Errno 2] No such file or directory: '/root/.cache/kagglehub/datasets/mohabenloughmari/miccai-task2-full/versions/MedViTV2-main'
/content
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


ModuleNotFoundError: No module named 'MedViT'

In [ ]:



class DualBranchMedViT(nn.Module):
    def __init__(self, num_classes, pretrained_weights=None, **kwargs):
        super().__init__()
        
        # Two separate MedViT encoders (or shared weights — your choice)
        self.branch_image = MedViT(num_classes=num_classes, **kwargs)
        self.branch_localizer = MedViT(num_classes=num_classes, **kwargs)
        
        # Remove both classification heads — we'll fuse before classifying
        final_dim = self.branch_image.proj_head[0].in_features  # e.g. 512
        self.branch_image.proj_head = nn.Identity()
        self.branch_localizer.proj_head = nn.Identity()
        
        # Fusion + classification head
        self.classifier = nn.Sequential(
            nn.Linear(final_dim * 2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
        
        if pretrained_weights:
            state_dict = torch.load(pretrained_weights, map_location='cpu')
            self.branch_image.load_state_dict(state_dict, strict=False)
            self.branch_localizer.load_state_dict(state_dict, strict=False)

    def forward(self, image, localizer):
        feat_img = self.branch_image(image)        # (B, final_dim)
        feat_loc = self.branch_localizer(localizer) # (B, final_dim)
        fused = torch.cat([feat_img, feat_loc], dim=1)  # (B, final_dim*2)
        return self.classifier(fused)

In [ ]:
NUM_CLASSES = df_train['label'].nunique()

model = DualBranchMedViT(
    num_classes=NUM_CLASSES,
    stem_chs=[64, 32, 64],
    depths=[2, 2, 6, 2],       # MedViT-Small (depths[2] must be divisible by 3)
    dims=[64, 128, 320, 512],
    path_dropout=0.2,
).to(device)


initialize_weights...
initialize_weights...


In [ ]:
optimizer = torch.optim.AdamW([
    {'params': model.branch_image.parameters(), 'lr': 1e-4},
    {'params': model.branch_localizer.parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 1e-3},  # much higher for the head
], weight_decay=0.05)

In [ ]:
import torch.nn.functional as F
class FocalLoss(nn.Module):
    def __init__(self, gamma=2):
        super().__init__()
        self.gamma = gamma
        
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean()

criterion = FocalLoss(gamma=2)

In [ ]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, labels in pbar:
            images, localisers, labels = images.to(device), localisers.to(device), labels.long().to(device)
            outputs = model(images, localisers)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    
    # Print classification report if requested
    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds, 
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")
    
    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1_score(all_labels, all_preds, average='micro'),
        'Rk-correlation': matthews_corrcoef(all_labels, all_preds),
        'Specificity': specificity(all_labels, all_preds),
        'Quadratic-weighted_Kappa': cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    }

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

num_epochs = 7
for epoch in range(num_epochs):
    model.train()
    all_preds = []
    all_labels = []
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, localizers, labels in train_bar:
        images = images.to(device)
        localizers = localizers.to(device)
        labels = labels.to(device).long()
        outputs = model(images, localizers)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nEpoch {epoch+1}/{num_epochs} — "
          f"Acc: {acc:.4f} | "
          f"F1: {f1_score(all_labels, all_preds, average='micro'):.4f} | "
          f"MCC: {matthews_corrcoef(all_labels, all_preds):.4f} | "
          f"Spec: {specificity(all_labels, all_preds):.4f} | "
          f"QWK: {cohen_kappa_score(all_labels, all_preds, weights='quadratic'):.4f}\n")
    
    # ADD THIS - Per-class performance
    print("Per-Class Performance:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Class 0', 'Class 1', 'Class 2'],
                                digits=4))
    print("-" * 80)

Epoch 1/7 [Train]: 100%|██████████| 674/674 [17:02<00:00,  1.52s/it, loss=0.4238]



Epoch 1/7 — Acc: 0.5486 | F1: 0.5486 | MCC: 0.3231 | Spec: 0.7743 | QWK: 0.4151

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.5581    0.5408    0.5493      2681
     Class 1     0.4916    0.4821    0.4868      2717
     Class 2     0.5938    0.6237    0.6084      2684

    accuracy                         0.5486      8082
   macro avg     0.5478    0.5489    0.5482      8082
weighted avg     0.5476    0.5486    0.5479      8082

--------------------------------------------------------------------------------


Epoch 2/7 [Train]: 100%|██████████| 674/674 [16:33<00:00,  1.47s/it, loss=0.1305]



Epoch 2/7 — Acc: 0.6592 | F1: 0.6592 | MCC: 0.4889 | Spec: 0.8298 | QWK: 0.6248

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.6669    0.6603    0.6636      2744
     Class 1     0.5676    0.5407    0.5538      2632
     Class 2     0.7323    0.7735    0.7523      2706

    accuracy                         0.6592      8082
   macro avg     0.6556    0.6582    0.6566      8082
weighted avg     0.6565    0.6592    0.6576      8082

--------------------------------------------------------------------------------


Epoch 3/7 [Train]: 100%|██████████| 674/674 [16:27<00:00,  1.46s/it, loss=0.3362]


...


Epoch 5/7 — Acc: 0.8330 | F1: 0.8330 | MCC: 0.7498 | Spec: 0.9164 | QWK: 0.8347

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8158    0.8430    0.8292      2732
     Class 1     0.7947    0.7456    0.7693      2673
     Class 2     0.8855    0.9100    0.8976      2677

    accuracy                         0.8330      8082
   macro avg     0.8320    0.8329    0.8320      8082
weighted avg     0.8319    0.8330    0.8320      8082

--------------------------------------------------------------------------------


Epoch 6/7 [Train]: 100%|██████████| 674/674 [16:29<00:00,  1.47s/it, loss=0.0543]



Epoch 6/7 — Acc: 0.8596 | F1: 0.8596 | MCC: 0.7896 | Spec: 0.9296 | QWK: 0.8621

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8430    0.8670    0.8548      2744
     Class 1     0.8290    0.7882    0.8081      2706
     Class 2     0.9062    0.9252    0.9156      2632

    accuracy                         0.8596      8082
   macro avg     0.8594    0.8601    0.8595      8082
weighted avg     0.8589    0.8596    0.8590      8082

--------------------------------------------------------------------------------


Epoch 7/7 [Train]: 100%|██████████| 674/674 [16:28<00:00,  1.47s/it, loss=0.6940]


Epoch 7/7 — Acc: 0.8857 | F1: 0.8857 | MCC: 0.8288 | Spec: 0.9428 | QWK: 0.8918

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8692    0.8895    0.8792      2697
     Class 1     0.8625    0.8212    0.8413      2712
     Class 2     0.9241    0.9473    0.9355      2673

    accuracy                         0.8857      8082
   macro avg     0.8853    0.8860    0.8854      8082
weighted avg     0.8851    0.8857    0.8851      8082

--------------------------------------------------------------------------------


In [ ]:
results = evaluate(model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0895    0.1228    0.1036       578
     Class 1     0.8379    0.8127    0.8251      4810
     Class 2     0.0996    0.0779    0.0874       321

    accuracy                         0.7015      5709
   macro avg     0.3424    0.3378    0.3387      5709
weighted avg     0.7207    0.7015    0.7106      5709


Accuracy: 70.15%
F1 Score: 0.7015
Quadratic Weighted Kappa: 0.0260
Rk Correlation (Spearman): -0.0118
Specificity: 0.6588


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

num_epochs = 4
for epoch in range(num_epochs):
    model.train()
    all_preds = []
    all_labels = []
    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
    for images, localizers, labels in train_bar:
        images = images.to(device)
        localizers = localizers.to(device)
        labels = labels.to(device).long()
        outputs = model(images, localizers)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_bar.set_postfix(loss=f"{loss.item():.4f}")

        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f"\nEpoch {epoch+1}/{num_epochs} — "
          f"Acc: {acc:.4f} | "
          f"F1: {f1_score(all_labels, all_preds, average='micro'):.4f} | "
          f"MCC: {matthews_corrcoef(all_labels, all_preds):.4f} | "
          f"Spec: {specificity(all_labels, all_preds):.4f} | "
          f"QWK: {cohen_kappa_score(all_labels, all_preds, weights='quadratic'):.4f}\n")
    
    # ADD THIS - Per-class performance
    print("Per-Class Performance:")
    print(classification_report(all_labels, all_preds, 
                                target_names=['Class 0', 'Class 1', 'Class 2'],
                                digits=4))
    print("-" * 80)

Epoch 1/4 [Train]: 100%|██████████| 674/674 [16:27<00:00,  1.47s/it, loss=0.6533]



Epoch 1/4 — Acc: 0.8802 | F1: 0.8802 | MCC: 0.8205 | Spec: 0.9401 | QWK: 0.8877

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8615    0.8833    0.8723      2691
     Class 1     0.8485    0.8183    0.8331      2697
     Class 2     0.9295    0.9391    0.9343      2694

    accuracy                         0.8802      8082
   macro avg     0.8798    0.8803    0.8799      8082
weighted avg     0.8798    0.8802    0.8799      8082

--------------------------------------------------------------------------------


Epoch 2/4 [Train]: 100%|██████████| 674/674 [16:25<00:00,  1.46s/it, loss=0.0388]



Epoch 2/4 — Acc: 0.9029 | F1: 0.9029 | MCC: 0.8544 | Spec: 0.9514 | QWK: 0.9067

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8915    0.9115    0.9014      2678
     Class 1     0.8826    0.8520    0.8670      2717
     Class 2     0.9338    0.9457    0.9397      2687

    accuracy                         0.9029      8082
   macro avg     0.9027    0.9031    0.9027      8082
weighted avg     0.9026    0.9029    0.9026      8082

--------------------------------------------------------------------------------


Epoch 3/4 [Train]: 100%|██████████| 674/674 [16:25<00:00,  1.46s/it, loss=0.0074]



Epoch 3/4 — Acc: 0.9171 | F1: 0.9171 | MCC: 0.8758 | Spec: 0.9585 | QWK: 0.9218

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.9012    0.9236    0.9122      2735
     Class 1     0.9017    0.8682    0.8846      2693
     Class 2     0.9486    0.9601    0.9543      2654

    accuracy                         0.9171      8082
   macro avg     0.9172    0.9173    0.9171      8082
weighted avg     0.9169    0.9171    0.9168      8082

--------------------------------------------------------------------------------


Epoch 4/4 [Train]: 100%|██████████| 674/674 [16:21<00:00,  1.46s/it, loss=0.1113]


Epoch 4/4 — Acc: 0.9134 | F1: 0.9134 | MCC: 0.8702 | Spec: 0.9567 | QWK: 0.9200

Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.8985    0.9231    0.9107      2744
     Class 1     0.8947    0.8568    0.8753      2618
     Class 2     0.9456    0.9581    0.9518      2720

    accuracy                         0.9134      8082
   macro avg     0.9129    0.9127    0.9126      8082
weighted avg     0.9131    0.9134    0.9131      8082

--------------------------------------------------------------------------------


In [ ]:
results = evaluate(model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.0812    0.0761    0.0786       578
     Class 1     0.8329    0.8613    0.8469      4810
     Class 2     0.0000    0.0000    0.0000       321

    accuracy                         0.7334      5709
   macro avg     0.3047    0.3125    0.3085      5709
weighted avg     0.7100    0.7334    0.7215      5709


Accuracy: 73.34%
F1 Score: 0.7334
Quadratic Weighted Kappa: 0.0054
Rk Correlation (Spearman): -0.0482
Specificity: 0.6476


In [ ]:
torch.save(model.state_dict(), 'model_weights.pt')
